# autofollowdown — full demo

A unified, simple toolkit for compressing AI models — **quantization, pruning, knowledge distillation** — with *real* (non-mock) benchmarks and a one-command flow.

This notebook shows everything `autofollowdown` can do, and explains **which benchmarks** it uses so you can trust the numbers.

Repo: https://github.com/peetwan/autofollowdown

## Setup

On Colab or a fresh machine, install first (uncomment the pip line).

In [1]:
# On Colab / a fresh machine, install first (uncomment):
# !pip install "git+https://github.com/peetwan/autofollowdown#egg=autofollowdown[examples]"
try:
    import autofollowdown as afd
except ImportError:
    # running from a clone (notebooks/ is inside the repo): add the repo root
    import os, sys
    sys.path.insert(0, os.path.abspath('..'))
    import autofollowdown as afd
print('autofollowdown', afd.__version__)

autofollowdown 0.1.0


## 1. What's inside — backends + benchmark catalog

`autofollowdown` always works with its built-in (native) engine, and can route to specialized libraries when they're installed.

In [2]:
from autofollowdown import all_backends
for b in all_backends():
    status = 'installed' if b.is_available() else f'optional  ({b.install_hint})'
    print(f'{b.name:40} {status}')

autofollowdown (native)                  installed
Microsoft NNI                            optional  (pip install nni)
llm-compressor (vLLM)                    optional  (pip install llmcompressor)
NVIDIA TensorRT Model Optimizer          optional  (pip install nvidia-modelopt)


## 2. Core API — compress a real model

`ModelCompressor` is chainable: prune, quantize, distill, export. Every step changes real weights. Here we show pruning creating real sparsity and INT8 quantization shrinking the model.

In [3]:
import io, torch, torch.nn as nn
from autofollowdown import ModelCompressor, count_parameters

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 16, 3, padding=1); self.relu = nn.ReLU()
        self.fc1 = nn.Linear(16*8*8, 128); self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = self.relu(self.conv(x)); x = torch.flatten(x, 1)
        return self.fc2(self.relu(self.fc1(x)))

def size_mb(m):
    b = io.BytesIO(); torch.save(m, b); return b.getbuffer().nbytes / 1e6

# Pruning -> real sparsity
m = Net(); _, _, s0 = count_parameters(m)
ModelCompressor(m).prune(sparsity=0.5, method='unstructured')
_, _, s1 = count_parameters(m)
print(f'unstructured prune: sparsity {s0:.0%} -> {s1:.0%}')

# Quantization -> smaller model
m2 = Net(); before = size_mb(m2)
q = ModelCompressor(m2).quantize(method='int8', approach='dynamic').model
print(f'int8 dynamic: {before:.3f} MB -> {size_mb(q):.3f} MB')

unstructured prune: sparsity 0% -> 50%
int8 dynamic: 0.534 MB -> 0.138 MB


## 3. ⭐ One command: compress every way, benchmark, and pick

`compress_and_benchmark` applies all methods, benchmarks them on real data, and recommends the best size/quality trade-off. We train a small CNN on the real scikit-learn `digits` dataset (no download).

In [4]:
import torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from autofollowdown import compress_and_benchmark

torch.manual_seed(0)
d = load_digits()
X = torch.tensor(d.images.astype('float32')/16.0).unsqueeze(1)
y = torch.tensor(d.target, dtype=torch.long)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(Xte, yte), batch_size=64)

class CNN(nn.Module):
    def __init__(self, w=32):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1,w,3,padding=1), nn.ReLU(),
                               nn.Conv2d(w,w,3,padding=1), nn.ReLU())
        self.c = nn.Sequential(nn.Flatten(), nn.Linear(w*8*8,128), nn.ReLU(),
                               nn.Linear(128,10))
    def forward(self, x): return self.c(self.f(x))

baseline = CNN()
opt = torch.optim.Adam(baseline.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss()
baseline.train()
for _ in range(6):
    for xb, yb in train_loader:
        opt.zero_grad(); lossf(baseline(xb), yb).backward(); opt.step()

study = compress_and_benchmark(baseline, eval_loader=test_loader)

[W606 21:38:30.133098000 qlinear_dynamic.cpp:252] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


In [5]:
from IPython.display import Markdown, display
display(Markdown(study.to_markdown()))
print('Recommended:', study.recommended)

| Model | Size (MB) | Params | Sparsity | Latency (ms) | Acc | Fidelity | Size× | Speed× | ΔAcc |
|---|---|---|---|---|---|---|---|---|---|
| baseline | 1.047 | 273,130 | 0.0% | 0.03 | 95.56% | 100.00% | — | — | — |
| int8 dynamic | 0.294 | 9,568 | 0.0% | 0.38 | 95.56% | 99.56% | 3.56× | 0.08× | +0.00% |
| prune 50% | 1.047 | 273,130 | 50.0% | 0.03 | 94.44% | 98.22% | 1.00× | 0.99× | -1.11% |
| prune+quantize | 0.294 | 9,568 | 17.8% | 0.38 | 94.44% | 98.22% | 3.56× | 0.08× | -1.11% |

Recommended: int8 dynamic


Pick a variant by name (or take the recommendation) and export it:

In [6]:
best_model = study.best()                 # the recommended model
# or: chosen = study.pick('int8 dynamic')
path = study.export(study.recommended, 'best_model.pt')
print('saved recommended variant ->', path)

saved recommended variant -> best_model.pt


## 4. Auto-picker — the best *library* for your model

`autofollowdown` profiles a model and ranks compression backends (native always on; NNI / llm-compressor / NVIDIA ModelOpt optional), telling you what's ideal and what's runnable right now.

In [7]:
from autofollowdown import explain
print(explain(baseline))   # our trained CNN

Model profile: ModelProfile(family=cnn, params=273,130, conv=True, transformer=False, hf=False, cuda=False)

Ranked compression backends:
 → 1. [0.60] autofollowdown (native): unstructured-0.5 + int8-dynamic (prune+quantize) — runnable
      Global magnitude pruning then portable INT8 — no GPU needed.
   2. [0.90] Microsoft NNI: L1FilterPruner + ModelSpeedup (structured-prune) — not installed
      Channel/filter pruning that ModelSpeedup turns into a genuinely smaller, faster model (real FLOP reduction).
      install: pip install nni
   3. [0.60] NVIDIA TensorRT Model Optimizer: INT8 PTQ (ptq) — not installed
      Calibrated PTQ (SmoothQuant/AWQ/NVFP4) via mtq.quantize, exportable to TensorRT. Requires an NVIDIA GPU.
      install: pip install nvidia-modelopt

Auto-pick (runnable now): autofollowdown (native)


## 5. Which benchmarks do we use?

Honest evaluation matters. Here is exactly what `autofollowdown` measures.

**Always measured (any model):** on-disk size (MB), parameter count, true sparsity, inference latency (p50), throughput, and — when you pass eval data — top-1 accuracy and output fidelity (agreement with the original model).

**Vision example:** the scikit-learn `digits` dataset (1,797 real 8x8 handwritten digits, ships with scikit-learn — no download). Bring your own DataLoader for your task.

**LLMs — the field-standard suite** (from GPTQ, AWQ, SparseGPT, LLMCBench, LeanQuant, NVIDIA MINITRON, Apple LLM-KICK):

| Pillar | Datasets / tasks | Measures |
|---|---|---|
| Perplexity ↓ | `WikiText-2`, `C4`, `PTB` | language-modeling quality |
| Commonsense (0-shot) | `arc_easy`, `arc_challenge`, `hellaswag`, `winogrande`, `piqa`, `boolq`, `openbookqa`, `lambada` | reasoning |
| Knowledge (5-shot) | `mmlu` | broad factual knowledge |
| Advanced | `mmlu_pro` | harder 10-choice reasoning |
| Multilingual | `mmlu_prox` / `mmlu_prox_lite` (29 languages) | cross-lingual reasoning |
| Reasoning | `gsm8k` | math word problems |
| Truthfulness | `truthfulqa` | reliability |

We compute **WikiText-2 perplexity** ourselves; the accuracy tasks run via EleutherAI's `lm-evaluation-harness`, and we generate the exact command for you.

In [8]:
from autofollowdown import STANDARD_LLM_TASKS, mmlu_prox_tasks, lm_eval_command

for group, tasks in STANDARD_LLM_TASKS.items():
    print(f'{group:22} {tasks}')

print()
# MMLU-ProX multilingual task ids (lite = 658 Q/language, fast for compressed models)
print('MMLU-ProX (en, th, zh):', mmlu_prox_tasks(['en','th','zh'], lite=True))

print()
# The exact command to run the full accuracy suite on a (compressed) model:
print(lm_eval_command('./best_model_hf', tasks=STANDARD_LLM_TASKS['commonsense_zeroshot'] + ['mmlu_prox_lite_en'], num_fewshot=0))

perplexity             ['wikitext2', 'c4', 'ptb']
commonsense_zeroshot   ['arc_easy', 'arc_challenge', 'hellaswag', 'winogrande', 'piqa', 'openbookqa', 'boolq', 'lambada_openai']
knowledge              ['mmlu']
advanced_knowledge     ['mmlu_pro']
multilingual           ['mmlu_prox_en', 'mmlu_prox_lite_en']
reasoning              ['gsm8k']
truthfulness           ['truthfulqa_mc2']

MMLU-ProX (en, th, zh): ['mmlu_prox_lite_en', 'mmlu_prox_lite_th', 'mmlu_prox_lite_zh']

lm_eval --model hf --model_args pretrained=./best_model_hf --tasks arc_easy,arc_challenge,hellaswag,winogrande,piqa,openbookqa,boolq,lambada_openai,mmlu_prox_lite_en --device cuda:0 --batch_size auto --num_fewshot 0


## 6. LLM perplexity benchmark

Perplexity (lower = better) is the headline metric for compressed LLMs. Below we demonstrate the metric on a tiny in-process GPT-2 (no download). For a real model on real WikiText-2, run the CLI in the next cell.

In [9]:
import torch, transformers
from autofollowdown import perplexity_from_ids

cfg = transformers.GPT2Config(vocab_size=128, n_positions=64, n_embd=32, n_layer=2, n_head=2)
tiny = transformers.GPT2LMHeadModel(cfg).eval()
ids = torch.randint(0, 128, (1, 160))
print('perplexity (tiny demo model):', round(perplexity_from_ids(tiny, ids, stride=32, max_length=64), 2))

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


perplexity (tiny demo model): 129.91


For a **real** LLM on **real** WikiText-2 (downloads a small model):

In [10]:
# !pip install datasets
# !autofollowdown benchmark-llm --model facebook/opt-125m
#
# Typical real result (OPT-125M, WikiText-2):
#   baseline fp32 : 477.8 MB,  perplexity 30.36
#   int8 dynamic  : 271.9 MB,  perplexity 31.87   (1.76x smaller)

## One-liners (after `pip install`)

```bash
autofollowdown auto                      # compress every way, benchmark, pick a winner
autofollowdown info                      # backends + benchmark catalog
autofollowdown benchmark-vision          # offline CNN benchmark
autofollowdown benchmark-llm             # LLM perplexity benchmark
autofollowdown autopick                  # best-library recommendation per model
```

That's the whole toolkit — happy compressing! 🎉